In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from ipywidgets import Dropdown, SelectMultiple, interactive_output, VBox, HBox, IntSlider, Layout, Label, HTML
from IPython.display import display, HTML as IPHTML
import warnings
warnings.filterwarnings("ignore")

In [21]:
# load dataset dataset_1980_2020.csv
crime_df = pd.read_csv('dataset_1980_2020.csv')
crime_df.head()

,Year,State,0 to 11,12 to 17,Male,Female,Unknown Gender,White,Black,Amer. Indian/Alaskan Native,...,Stranger,Unknown,Firearm,Knife,Blunt object,Personal,Other/unknown Weapon,One offender involved,Two or more offenders involved,Unknown number of offenders involved
0,1980,AL,0,18,13,5,0,7,10,1,...,4,0,15,3,0,0,0,15,3,0
1,1981,AL,0,21,19,1,0,13,8,0,...,5,0,13,8,0,0,0,11,10,0
2,1982,AL,1,19,15,4,0,5,14,0,...,6,0,7,5,2,2,3,13,6,0
3,1983,AL,2,20,21,1,0,7,15,0,...,4,0,13,7,0,1,1,16,6,0
4,1984,AL,0,21,19,2,0,7,13,0,...,5,2,12,3,0,4,1,10,10,0


In [23]:
def plot_crime_map(df, filters, year=None):
    # filter by year
    if year is not None:
        df = df[df['Year'] == year]

    valid_filters = [f for f in filters if f in df.columns]

    if not valid_filters:
        raise ValueError("no valid filters provided")
    
    # sum accross the selecter filter columns
    df['Filtered Crime Count'] = df[valid_filters].sum(axis=1)
    state_crime = df.groupby('State')['Filtered Crime Count'].sum().reset_index()

    # plot
    fig = px.choropleth(
        state_crime,
        locations='State',
        locationmode='USA-states',
        color='Filtered Crime Count',
        scope="usa",
        color_continuous_scale="Reds",
        labels={'Filtered Crime Count': 'Crime Count'},
        title=f"Crime Map - Filters: {', '.join(filters)}" + (f" ({year})" if year else "")
    )
    fig.show()

def interactive_crime_map(crime_df):
    all_filters = [
        '0 to 11', '12 to 17',
        'Male', 'Female', 'Unknown Gender',
        'White', 'Black', 'Amer. Indian/Alaskan Native', 'Asian/Nat. Hawaiian/Pac Isl', 'Unknown Race',
        'Family', 'Acquaintance', 'Stranger', 'Unknown',
        'Firearm', 'Knife', 'Blunt object', 'Personal', 'Other/unknown Weapon',
        'One offender involved', 'Two or more offenders involved', 'Unknown number of offenders involved'
    ]

    year_options = crime_df['Year'].dropna().unique()
    filter_label = HTML(
        value="<b>Select Filters</b>",
        layout=Layout(margin='0 0 10px 0')
    )  
    year_slider = IntSlider(
        value=int(year_options.min()),
        min=int(year_options.min()),
        max=int(year_options.max()),
        step=1,
        description='Year:',
        continuous_update=False,
        layout=Layout(width='100%', margin='20px 0 0 0')
    )
    year_slider.style = {
        'description_width': '80px',
        'handle_color': '#d62728',
        'font_size': '16px'
    }
    filter_select = SelectMultiple(
        options=all_filters,
        value=('Male',),
        rows=25,
        description = '',
        layout=Layout(
            width='100%',
            height='auto',
            overflow_y='visible',
            white_space='normal'
        )
    )
    filter_select.style = {'description_width': '0px'}


    left_panel = VBox([filter_label, filter_select], layout=Layout(width='300px', height='100%', align_items='stretch'))
    display(IPHTML("<style>.widget-readout { display: none !important; }</style>"))
    
    out = interactive_output(
        lambda filters, year: plot_crime_map(crime_df, list(filters), year),
        {'filters': filter_select, 'year': year_slider}
    )

    right_panel = VBox(
        [out, year_slider],
        layout=Layout(flex='1', height='100%', align_items='stretch')
    )
    full_ui = HBox(
        [left_panel, right_panel],
        layout=Layout(width='100%', height='auto', align_items='flex-start')
    )
    display(full_ui)

interactive_crime_map(crime_df)
